In [12]:
import pandas as pd
import csv

In [13]:
def parse_params(params_str):
    params = {}
    for item in params_str.split():
        if '=' in item:
            key, value = item.split('=', 1)
            params[key] = value
    return params

# aggregates results over all parameters except those in params_to_compare
def results_csv_to_df(csv_file, params_to_aggregate=None):
    results = []

    with open(csv_file, newline='') as f:
        reader = csv.DictReader(f, fieldnames=[
            'accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed', 'not_fully_unmasked',
            'checkpoint', 'strategy', 'params'
        ])
        reader.__next__()  # Skip header row
        for row in reader:
            params = parse_params(row['params'])
            row_dict = {
                'accuracy': float(row['accuracy']),
                'correctly_filled_cells': float(row['correctly_filled_cells']),
                'checkpoint': row['checkpoint'],
                'strategy': row['strategy'],
                'nfe': float(row['nfe']),
                'time': float(row['time']),
                'speed': float(row['speed']),
            }
            row_dict.update(params.items())
            results.append(row_dict)
    
    results_df = pd.DataFrame(results)
    print(f"Total rows before aggregation: {len(results_df)}")
    
    # Aggregate the accuracy, correctly_filled_cells, nfe, time, and speed over the rows where all parameters except those in params_to_aggregate are the same
    if params_to_aggregate is not None:
        groupby_cols = [col for col in results_df.columns if col not in params_to_aggregate + ['accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed']]
    else:
        groupby_cols = [col for col in results_df.columns if col not in ['accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed']]
    
    aggregated_df = results_df.groupby(groupby_cols, dropna=False).agg(
        accuracy_mean=pd.NamedAgg(column='accuracy', aggfunc='mean'),
        accuracy_std=pd.NamedAgg(column='accuracy', aggfunc='std'),
        correctly_filled_cells_mean=pd.NamedAgg(column='correctly_filled_cells', aggfunc='mean'),
        correctly_filled_cells_std=pd.NamedAgg(column='correctly_filled_cells', aggfunc='std'),
        nfe=pd.NamedAgg(column='nfe', aggfunc='mean'),
        time=pd.NamedAgg(column='time', aggfunc='mean'),
        speed=pd.NamedAgg(column='speed', aggfunc='mean'),
    ).reset_index()

    print(f"Rows after aggregation: {len(aggregated_df)}")
    return aggregated_df

In [14]:
params_to_aggregate = ['seed']
df = results_csv_to_df('results_presentation_independent.csv', params_to_aggregate=params_to_aggregate)
df

Total rows before aggregation: 24
Rows after aggregation: 8


,checkpoint,strategy,unmask_token,dataset,num_samples,num_denoising_steps,num_self_correction_steps,time_steps,batch_size,min_p,compile_torch,do_beam_search,change_token,accuracy_mean,accuracy_std,correctly_filled_cells_mean,correctly_filled_cells_std,nfe,time,speed
0,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,easy,6400,80,0,fixed,64,0,0,false,change_categorical,0.228133,0.004735,0.771033,0.000981,1.24,254.000000,25.183333
1,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,easy,6400,80,0,fixed,64,0,0,false,change_max,0.273467,0.003907,0.794733,0.000971,1.24,270.333333,23.963333
2,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,hard,6400,80,0,fixed,64,0,0,false,change_categorical,0.128400,0.003027,0.670400,0.001153,1.24,250.333333,25.583333
3,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,hard,6400,80,0,fixed,64,0,0,false,change_max,0.147200,0.002600,0.685867,0.000945,1.24,248.666667,25.740000
4,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_categorical,easy,6400,80,0,fixed,64,0,0,false,NaN,0.102800,0.003051,0.647800,0.001058,1.24,123.666667,51.900000
5,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_categorical,hard,6400,80,0,fixed,64,0,0,false,NaN,0.060033,0.002386,0.582700,0.001353,1.24,124.000000,51.540000
6,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_max,easy,6400,80,0,fixed,64,0,0,false,NaN,0.176967,0.003262,0.697667,0.002043,1.24,125.000000,51.230000
7,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_max,hard,6400,80,0,fixed,64,0,0,false,NaN,0.098433,0.001872,0.619300,0.002476,1.24,131.666667,48.996667


In [15]:
# create a column "token_update" that is "max" if unmask_token is MDM_max or if change_token is change_max, "categorical" if unmask_token is MDM_categorical or if change_token is change_categorical, and "none" if both are none
def determine_token_update(row):
    if row['unmask_token'] == 'MDM_max' or row['change_token'] == 'change_max':
        return 'max'
    elif row['unmask_token'] == 'MDM_categorical' or row['change_token'] == 'change_categorical':
        return 'categorical'
    else:
        return 'none'
df['token_update'] = df.apply(determine_token_update, axis=1)
df

,checkpoint,strategy,unmask_token,dataset,num_samples,num_denoising_steps,num_self_correction_steps,time_steps,batch_size,min_p,...,do_beam_search,change_token,accuracy_mean,accuracy_std,correctly_filled_cells_mean,correctly_filled_cells_std,nfe,time,speed,token_update
0,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,easy,6400,80,0,fixed,64,0,...,false,change_categorical,0.228133,0.004735,0.771033,0.000981,1.24,254.000000,25.183333,categorical
1,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,easy,6400,80,0,fixed,64,0,...,false,change_max,0.273467,0.003907,0.794733,0.000971,1.24,270.333333,23.963333,max
2,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,hard,6400,80,0,fixed,64,0,...,false,change_categorical,0.128400,0.003027,0.670400,0.001153,1.24,250.333333,25.583333,categorical
3,checkpoints/gidd_0_2/300_epochs,gidd_independent_positions_decomposed_update_d...,NaN,hard,6400,80,0,fixed,64,0,...,false,change_max,0.147200,0.002600,0.685867,0.000945,1.24,248.666667,25.740000,max
4,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_categorical,easy,6400,80,0,fixed,64,0,...,false,NaN,0.102800,0.003051,0.647800,0.001058,1.24,123.666667,51.900000,categorical
5,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_categorical,hard,6400,80,0,fixed,64,0,...,false,NaN,0.060033,0.002386,0.582700,0.001353,1.24,124.000000,51.540000,categorical
6,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_max,easy,6400,80,0,fixed,64,0,...,false,NaN,0.176967,0.003262,0.697667,0.002043,1.24,125.000000,51.230000,max
7,checkpoints/mdlm/300_epochs,mdlm_vanilla,MDM_max,hard,6400,80,0,fixed,64,0,...,false,NaN,0.098433,0.001872,0.619300,0.002476,1.24,131.666667,48.996667,max


In [17]:
grouped_df = df.groupby(['checkpoint', 'strategy', 'token_update', 'dataset'], dropna=False)['accuracy_mean'].mean().unstack()#.sort_values(by='accuracy_mean')
grouped_df

dataset                                                                                              easy  \
checkpoint                      strategy                                           token_update             
checkpoints/gidd_0_2/300_epochs gidd_independent_positions_decomposed_update_di... categorical   0.228133   
                                                                                   max           0.273467   
checkpoints/mdlm/300_epochs     mdlm_vanilla                                       categorical   0.102800   
                                                                                   max           0.176967   

dataset                                                                                              hard  
checkpoint                      strategy                                           token_update            
checkpoints/gidd_0_2/300_epochs gidd_independent_positions_decomposed_update_di... categorical   0.128400  
                                                                                   max           0.147200  
checkpoints/mdlm/300_epochs     mdlm_vanilla                                       categorical   0.060033  
                                                                                   max           0.098433